# AIND Ephys Spike Sorting (Kilosort4)

Run [aind-ephys-spikesort-kilosort4](https://github.com/AllenNeuralDynamics/aind-ephys-spikesort-kilosort4/tree/main/code) on a preprocessed version of the toy recording.

Kilosort4 fits whitening / template models from the data and refuses to run on a 1-second / 16-channel slice (the covariance is ill-conditioned and `torch.linalg.svd` fails). To exercise the full sorting pipeline end-to-end, this notebook **re-runs preprocessing with an 8-second window** (rather than reusing the 1-second `preprocessing_results/` folder from `2_aind_ephys_preprocessing.ipynb`), then runs the KS4 capsule against that output.

The capsule expects `../data/` (relative to its `code/`) to contain `preprocessed_<name>/` folders and matching `binary_<name>.json` files. Outputs (sortings, logs, processing metadata) land in `../results/`.

Note: KS4 is a GPU sorter. On Apple Silicon `torch_device='cpu'` works but is slow; on the toy data KS4 typically detects very few or zero units — the goal here is pipeline plumbing, not biology.

## Imports

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

## Install Kilosort4 + capsule deps

`kilosort` pulls in `torch`. `aind-data-schema` is needed for the processing-metadata records the capsule writes.

In [2]:
subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "kilosort", "aind-data-schema",
    ],
    check=True,
)

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Resolved 44 packages in 20ms
Uninstalled 3 packages in 32ms
Installed 3 packages in 8ms
 - pydantic==2.12.5
 + pydantic==2.11.10
 - pydantic-core==2.41.5
 + pydantic-core==2.33.2
 - setuptools==82.0.0
 + setuptools==81.0.0


CompletedProcess(args=['uv', 'pip', 'install', '--python', '/Users/james/Documents/obi/code/obi-main/obi-one/.venv/bin/python', 'kilosort', 'aind-data-schema'], returncode=0)

## Clone the capsule and seed `data/` with the preprocessing outputs

In [3]:
ks_repo = Path("/tmp/aind-ephys-spikesort-kilosort4")
if not ks_repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth=1",
            "https://github.com/AllenNeuralDynamics/aind-ephys-spikesort-kilosort4.git",
            str(ks_repo),
        ],
        check=True,
    )

data_dir = ks_repo / "data"
results_dir = ks_repo / "results"
data_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

for stale in list(data_dir.iterdir()) + list(results_dir.iterdir()):
    if stale.is_dir():
        shutil.rmtree(stale)
    else:
        stale.unlink()

# Re-run preprocessing on an 8-second window so KS4 has enough data for whitening.
preproc_repo = Path("/tmp/aind-ephys-preprocessing")
assert preproc_repo.exists(), (
    f"{preproc_repo} not found. Run 2_aind_ephys_preprocessing.ipynb first."
)

preproc_results = preproc_repo / "results"
for stale in list(preproc_results.iterdir()):
    if stale.is_dir():
        shutil.rmtree(stale)
    else:
        stale.unlink()

argv = [
    "python", "-u", "run_capsule.py",
    "--params", "params_toy.json",
    "--t-start", "0",
    "--t-stop", "8",
    "--n-jobs", "1",
]
print("Re-running preprocessing with --t-stop 8:", " ".join(argv))
result = subprocess.run(argv, cwd=preproc_repo / "code", capture_output=True, text=True, check=False)
if result.returncode != 0:
    print(result.stdout)
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"preprocessing run failed with code {result.returncode}")
print("Preprocessing OK")

# Seed the KS4 data folder with `preprocessed_*` dirs and `binary_*.json` files.
for entry in preproc_results.iterdir():
    dest = data_dir / entry.name
    if entry.is_dir():
        shutil.copytree(entry, dest)
    else:
        shutil.copy2(entry, dest)

print("Seeded KS4 data dir:")
for p in sorted(data_dir.iterdir()):
    print(" ", p.name)

Cloning into '/tmp/aind-ephys-spikesort-kilosort4'...


Re-running preprocessing with --t-stop 8: python -u run_capsule.py --params params_toy.json --t-start 0 --t-stop 8 --n-jobs 1


Preprocessing OK
Seeded KS4 data dir:
  binary_block0_None_recording1.json
  data_process_preprocessing_block0_None_recording1.json
  preprocessed_block0_None_recording1
  preprocessed_block0_None_recording1.json
  preprocessedviz_block0_None_recording1.json


## Write a toy params.json

Starts from the upstream defaults but disables Kilosort's internal drift correction (`do_correction: false`) — the toy recording is too short and too small for drift estimation.

In [4]:
upstream_params = json.loads((ks_repo / "code" / "params.json").read_text())
params = {
    **upstream_params,
    "job_kwargs": {**upstream_params.get("job_kwargs", {}), "n_jobs": 1},
    # When --params is passed, top-level keys override the CLI flags. Set
    # skip_motion_correction here so the capsule won't try to read motion ops.
    "skip_motion_correction": True,
    "min_drift_channels": 64,
    "sorter": {
        **upstream_params.get("sorter", {}),
        "do_correction": False,   # toy data is too small for drift correction
        "nblocks": 0,
        "torch_device": "cpu",
        "whitening_range": 16,    # match num_channels of the toy probe
        "nskip": 1,               # use every batch (we only have a few)
        "nearest_chans": 8,
        "nearest_templates": 16,
    },
}
params_path = ks_repo / "code" / "params_toy.json"
params_path.write_text(json.dumps(params, indent=2))
print("Wrote", params_path)

Wrote /tmp/aind-ephys-spikesort-kilosort4/code/params_toy.json


## Run the spike-sorting capsule

We additionally pass `--skip-motion-correction` (motion was disabled in preprocessing) and `--n-jobs 1`.

In [5]:
argv = [
    "python", "-u", "run_capsule.py",
    "--params", params_path.name,
    "--skip-motion-correction",
    "--n-jobs", "1",
]
print("Running:", " ".join(argv))
result = subprocess.run(
    argv,
    cwd=ks_repo / "code",
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError(f"kilosort4 run failed with code {result.returncode}")

Running: python -u run_capsule.py --params params_toy.json --skip-motion-correction --n-jobs 1




SPIKE SORTING WITH KILOSORT4

	RAISE_IF_FAILS: True
	SKIP_MOTION_CORRECTION: True
	MIN_DRIFT_CHANNELS: 64
	N_JOBS: -1
Sorting recording: block0_None_recording1
Loading recording from binary JSON
BinaryFolderRecording: 70 channels - 30.0kHz - 1 segments - 240,001 samples - 8.00s 
                       float32 dtype - 64.09 MiB
Drift correction disabled
	Raw sorting output: KiloSortSortingExtractor: 14 units - 1 segments - 30.0kHz
	Sorting output without empty units: UnitsSelectionSorting: 14 units - 1 segments - 30.0kHz
	Saving results to ../results/spikesorted_block0_None_recording1
SPIKE SORTING time: 9.28s



## Copy outputs next to the notebook

In [6]:
local_results_dir = Path.cwd() / "spikesort_results"
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
shutil.copytree(results_dir, local_results_dir)

for p in sorted(local_results_dir.rglob("*")):
    rel = p.relative_to(local_results_dir)
    kind = "dir" if p.is_dir() else f"{p.stat().st_size} bytes"
    print(f"  {rel}  ({kind})")

  data_process_spikesorting_block0_None_recording1.json  (2889 bytes)
  spikesorted_block0_None_recording1  (dir)
  spikesorted_block0_None_recording1/numpysorting_info.json  (110 bytes)
  spikesorted_block0_None_recording1/properties  (dir)
  spikesorted_block0_None_recording1/properties/Amplitude.npy  (240 bytes)
  spikesorted_block0_None_recording1/properties/ContamPct.npy  (240 bytes)
  spikesorted_block0_None_recording1/properties/KSLabel.npy  (352 bytes)
  spikesorted_block0_None_recording1/properties/KSLabel_repeat.npy  (352 bytes)
  spikesorted_block0_None_recording1/properties/original_cluster_id.npy  (240 bytes)
  spikesorted_block0_None_recording1/provenance.json  (44094 bytes)
  spikesorted_block0_None_recording1/si_folder.json  (19197 bytes)
  spikesorted_block0_None_recording1/spikeinterface_log.json  (191 bytes)
  spikesorted_block0_None_recording1/spikes.npy  (47784 bytes)


## Load the sorting result

In [7]:
import spikeinterface.full as si

sorting_dirs = [p for p in local_results_dir.iterdir() if p.is_dir() and p.name.startswith("spikesorted_")]
print("Sorting dirs:", [p.name for p in sorting_dirs])

if sorting_dirs:
    sorting = si.load(sorting_dirs[0])
    print(sorting)
    print("num_units:", len(sorting.unit_ids))
    print("unit_ids:", list(sorting.unit_ids))

Sorting dirs: ['spikesorted_block0_None_recording1']
NumpyFolder (NumpyFolderSorting): 14 units - 1 segments - 30.0kHz
num_units: 14
unit_ids: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
